# 技能0 · Day 3 上机：描述统计与推断统计

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **numpy + pandas** 计算描述统计（均值/中位数/方差/分位数），理解客单价的右偏分布
2. 用 **scipy.stats** 执行 t 检验和卡方检验，判断 A/B 测试和用户分群的统计显著性
3. 用 **matplotlib + seaborn** 可视化营销数据的分布和关系
4. 计算 95% 置信区间，区分统计显著性与商业显著性
5. 用 Beta-Binomial 模型实现贝叶斯统计入门，对比频率派与贝叶斯派

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：numpy + pandas（描述统计）+ scipy.stats（假设检验）+ matplotlib/seaborn（可视化）。
营销映射：1000条A/B测试数据（旧版/新版落地页），从描述统计到贝叶斯推断的完整分析链。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ 所有库均为标准科学计算库，通常已随 conda/venv 安装。
> scipy.stats 是假设检验的权威实现，结果与 R/Stata 一致。

In [ ]:
# !pip install numpy pandas scipy matplotlib seaborn -q

## 1. 数据集背景与营销映射

**处理对象**：1000 条模拟真实营销场景的 A/B 测试数据。

| 字段 | 说明 | 示例 |
|------|------|------|
| user_id | 用户唯一标识 | 1-1000 |
| group | 实验分组（A=旧版落地页，B=新版落地页） | 'A' / 'B' |
| converted | 是否转化（1=购买，0=未购买） | 0 / 1 |
| spend | 消费金额（元，未转化为0） | 0.0 / 89.50 |
| segment | 用户分群（new/returning/vip） | 'new' |
| category | 购买品类（beauty/electronics/fitness/home） | 'beauty' |

**数据生成逻辑**：
- A组（旧版）：真实转化率约3.0%，客单价均值约33元（lognormal右偏分布）
- B组（新版）：真实转化率约6.0%，客单价均值约45元（lognormal右偏分布）

**营销映射**：在真实项目中，这些数据来自A/B测试平台API或业务数据库。统计分析的目标是从数据中提取决策依据：新版落地页是否真的更好？效果有多大？是否值得全量上线？

**理论连接**：假设检验是A/B测试的理论基础。p值告诉你"有没有效果"，效应量和CI告诉你"效果多大"，贝叶斯后验告诉你"转化率在某个区间的概率"。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# ============================================================
# 营销 A/B 测试数据（模拟真实电商场景）
# A组（旧版落地页）vs B组（新版落地页）
# ============================================================
n = 1000
group = np.random.choice(['A', 'B'], size=n, p=[0.5, 0.5])
# A组真实转化率~3.0%，B组真实转化率~6.0%
converted = np.where(group == 'A',
                     np.random.binomial(1, 0.030, n),
                     np.random.binomial(1, 0.060, n))
# 消费金额（转化者才有，lognormal分布模拟右偏客单价）
spend = np.where(converted == 1,
                 np.where(group == 'A',
                          np.random.lognormal(3.5, 0.5, n).round(2),
                          np.random.lognormal(3.8, 0.5, n).round(2)),
                 0.0)
# 用户分群
segment = np.random.choice(['new', 'returning', 'vip'], size=n, p=[0.4, 0.45, 0.15])
# 购买品类
category = np.random.choice(['beauty', 'electronics', 'fitness', 'home'], size=n, p=[0.3, 0.25, 0.2, 0.25])

df = pd.DataFrame({
    'user_id': range(1, n + 1),
    'group': group,
    'converted': converted,
    'spend': spend,
    'segment': segment,
    'category': category
})

print("营销 A/B 测试数据集")
print(f"总用户数: {len(df)}")
print(f"A组: {len(df[df.group=='A'])}人, B组: {len(df[df.group=='B'])}人")
print(f"总转化数: {df['converted'].sum()}")
print(f"\n前5行预览:")
print(df.head())

## 1：描述统计 -- A/B 两组客单价分布

**任务**：用 numpy + pandas 计算 A/B 两组的描述统计。

**营销场景**：你需要向营销总监汇报新旧两版落地页的客单价（AOV）对比。均值受异常值影响，中位数更稳健。偏度 > 0 说明右偏分布（少数大额消费拉高均值），这在消费数据中极为常见。

**理论要点**：
- 均值 vs 中位数：右偏分布下中位数 < 均值，中位数更能反映"典型用户"
- 方差/标准差：衡量消费的波动范围
- 分位数（Q1/Q3）：IQR = Q3 - Q1，衡量中间50%数据的离散程度
- 偏度（skewness）：> 0 为右偏，消费金额、广告花费等商业数据常见

In [ ]:
# 1. 用 numpy + pandas 计算 A/B 两组的描述统计
spend_a = df[df['group'] == 'A']['spend']
spend_b = df[df['group'] == 'B']['spend']

stats_summary = pd.DataFrame({
    'A组(旧版)': [spend_a.mean(), spend_a.median(), spend_a.var(), spend_a.std(),
                spend_a.quantile(0.25), spend_a.quantile(0.75), spend_a.skew()],
    'B组(新版)': [spend_b.mean(), spend_b.median(), spend_b.var(), spend_b.std(),
                spend_b.quantile(0.25), spend_b.quantile(0.75), spend_b.skew()]
}, index=['均值', '中位数', '方差', '标准差', 'Q1(25%)', 'Q3(75%)', '偏度'])

conv_rate_a = df[df['group'] == 'A']['converted'].mean()
conv_rate_b = df[df['group'] == 'B']['converted'].mean()

print("A/B 两组描述统计对比：")
print(stats_summary.round(4))
print(f"\nA组转化率: {conv_rate_a*100:.2f}%")
print(f"B组转化率: {conv_rate_b*100:.2f}%")
print(f"转化率差: {(conv_rate_b - conv_rate_a)*100:.2f}个百分点")
print(f"\n解读：B组客单价均值({spend_b.mean():.2f})高于A组({spend_a.mean():.2f})，")
print(f"      但偏度均为正(右偏)，说明少数大额消费拉高了均值。")
print(f"      中位数比均值更稳健地反映'典型用户'的消费水平。")

## 2. 概率分布与可视化理论

### 三大分布的营销应用

| 分布 | 适用场景 | 营销应用 |
|------|---------|---------|
| 正态分布 | 中心极限定理：大量独立变量和的分布 | A/B测试中转化率差的分布近似正态 |
| 二项分布 | n次伯努利试验的成功次数 | 广告点击（点/不点）、转化（买/不买） |
| 泊松分布 | 单位时间事件发生次数 | 客服来电、网站访问、购买次数 |

### 可视化的决策价值
- **直方图**：看分布形态（右偏？对称？多峰？）
- **箱线图**：看离散程度和异常值（IQR外的点）
- **散点图**：看两个变量的关系（转化与消费的关系）
- **柱状图**：比分群转化率（哪个群体转化最高？）

## 2：数据可视化 -- 营销转化分布

**任务**：用 matplotlib + seaborn 绘制 4 个子图。

**营销场景**：你需要为营销周报制作一页可视化看板，让非技术的市场团队直观理解 A/B 测试结果。

**要求**：
1. 直方图：A/B 两组转化者的客单价分布对比
2. 箱线图：A/B 两组客单价的离散程度和异常值
3. 散点图：用户消费分布（按转化状态着色）
4. 柱状图：各用户分群的转化率对比

In [ ]:
# 2. 用 matplotlib + seaborn 绘制可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 子图1: 直方图 - A/B两组消费分布（仅转化者）
spend_a_pos = spend_a[spend_a > 0]
spend_b_pos = spend_b[spend_b > 0]
axes[0, 0].hist(spend_a_pos, bins=20, alpha=0.6, label='A组(旧版)', color='#3498db', edgecolor='white')
axes[0, 0].hist(spend_b_pos, bins=20, alpha=0.6, label='B组(新版)', color='#e74c3c', edgecolor='white')
axes[0, 0].set_xlabel('消费金额 (元)')
axes[0, 0].set_ylabel('频次')
axes[0, 0].set_title('客单价分布对比（转化者）')
axes[0, 0].legend()

# 子图2: 箱线图 - A/B两组消费分布
bp = axes[0, 1].boxplot([spend_a_pos, spend_b_pos], labels=['A组(旧版)', 'B组(新版)'],
                         patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#3498db')
bp['boxes'][1].set_facecolor('#e74c3c')
axes[0, 1].set_ylabel('消费金额 (元)')
axes[0, 1].set_title('客单价箱线图（转化者）')

# 子图3: 散点图 - 用户消费分布
colors = df['converted'].map({0: 'lightgray', 1: '#e74c3c'})
axes[1, 0].scatter(df['user_id'], df['spend'], c=colors, alpha=0.5, s=10)
axes[1, 0].set_xlabel('用户编号')
axes[1, 0].set_ylabel('消费金额 (元)')
axes[1, 0].set_title('用户消费分布（红=转化，灰=未转化）')

# 子图4: 柱状图 - 各用户分群转化率
seg_conv = df.groupby('segment')['converted'].mean()
axes[1, 1].bar(seg_conv.index, seg_conv.values * 100, color=['#2ecc71', '#f39c12', '#9b59b6'], alpha=0.8)
axes[1, 1].set_ylabel('转化率 (%)')
axes[1, 1].set_title('各用户分群转化率')
for i, v in enumerate(seg_conv.values):
    axes[1, 1].text(i, v * 100 + 0.1, f'{v*100:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('marketing_ab_visualization.png', dpi=150)
plt.show()
print("图表已保存为 marketing_ab_visualization.png")

## 3. 假设检验理论

### 法庭审判类比

| 统计概念 | 法庭类比 | A/B测试场景 |
|---------|---------|-------------|
| 原假设 H₀ | 被告无罪 | 新方案和旧方案没有差异 |
| 备择假设 H₁ | 被告有罪 | 新方案更好 |
| p值 | 无罪前提下看到当前证据的概率 | H₀成立时观察到当前差异的概率 |
| 第一类错误 α | 冤枉好人 | 方案无效但判为有效 |
| 第二类错误 β | 放过坏人 | 方案有效但判为无效 |
| 统计功效 1-β | 正确判有罪的概率 | 正确识别有效方案的概率 |

### 关键认知
p值不告诉你"新方案有多好"，只告诉你"有没有效果"。p < 0.05 只是"有统计显著性"，不等于"有商业意义"。效果大小需要看效应量（effect size）和置信区间。

## 3：t 检验 -- A/B 两组转化率差异显著性

**任务**：用 scipy.stats.ttest_ind 执行单侧 t 检验。

**营销场景**：数据团队需要判断新版落地页的转化率是否显著高于旧版。如果显著，建议全量上线。

**假设**：
- H₀: B组转化率 <= A组转化率（新版不优于旧版）
- H₁: B组转化率 > A组转化率（新版更优）
- 显著性水平：α = 0.05

**提示**：对二值转化数据（0/1），ttest_ind 等价于两比例Z检验（大样本下）。`alternative='greater'` 表示单侧检验。

In [ ]:
# 3. 用 scipy.stats.ttest_ind 执行 t 检验
conv_a = df[df['group'] == 'A']['converted'].values
conv_b = df[df['group'] == 'B']['converted'].values

# 单侧 t 检验：H0: B<=A, H1: B>A
t_stat, p_value = stats.ttest_ind(conv_b, conv_a, alternative='greater')

print("--- t检验结果 ---")
print(f"  H0: B组转化率 <= A组转化率")
print(f"  H1: B组转化率 > A组转化率")
print(f"  A组转化率: {conv_a.mean()*100:.2f}% (n={len(conv_a)})")
print(f"  B组转化率: {conv_b.mean()*100:.2f}% (n={len(conv_b)})")
print(f"  t统计量: {t_stat:.4f}")
print(f"  p值: {p_value:.4f}")
print(f"  显著性水平: α = 0.05")
if p_value < 0.05:
    print(f"  结论: 拒绝原假设。B组转化率显著高于A组。")
    print(f"  建议: 全量上线B版落地页。")
else:
    print(f"  结论: 无法拒绝原假设。证据不足以证明B组更好。")
    print(f"  建议: 继续实验或保持A版。")
print(f"\n  注意: p值={p_value:.4f}，{'< 0.05 有统计显著性' if p_value < 0.05 else '>= 0.05 无统计显著性'}")
print(f"  但统计显著性 ≠ 商业显著性，需结合效应量和CI判断。")

## 4. 置信区间理论

### 95%置信区间的含义
如果重复实验100次，95次实验的CI会包含真实参数值。CI不仅告诉你"有没有效"（是否包含0），还告诉你"效果多大"（区间位置）和"估计多精确"（区间宽度）。

### 统计显著性 vs 商业显著性
当样本量很大时，即使0.01%的转化率差也可能"统计显著"。但0.01%的差异可能不值得全量上线。决策时需同时考虑：
- p值（有没有效果）
- 效应量（效果多大）
- 置信区间（估计精度）
- 商业ROI（投入产出比）

## 4：置信区间 -- 转化率差的 95% CI

**任务**：计算 B组与A组转化率差的 95% 置信区间。

**营销场景**：营销总监问"新版到底能提升多少转化率？"你不能只给一个点估计，要给出一个区间，并说明最保守情况下的提升是多少。

**方法**：正态近似（Wilson score区间）
- 标准误 SE = sqrt(p_pooled * (1-p_pooled) * (1/n_a + 1/n_b))
- 95% CI = 观察到的差 ± 1.96 * SE

**判断**：
- CI 不包含 0 → 有统计显著性
- CI 下界 > 业务最小有效阈值（如0.5%）→ 有商业显著性

In [ ]:
# 4. 计算转化率差的 95% 置信区间
n_a = len(conv_a)
n_b = len(conv_b)
obs_diff = conv_rate_b - conv_rate_a

# 合并比例和标准误（正态近似）
pooled_p = (conv_a.sum() + conv_b.sum()) / (n_a + n_b)
se = np.sqrt(pooled_p * (1 - pooled_p) * (1/n_a + 1/n_b))

# 95% 置信区间
z_alpha = stats.norm.ppf(0.975)
ci_lower = obs_diff - z_alpha * se
ci_upper = obs_diff + z_alpha * se

print("--- 95%置信区间 ---")
print(f"  A组转化率: {conv_rate_a*100:.2f}% (n={n_a})")
print(f"  B组转化率: {conv_rate_b*100:.2f}% (n={n_b})")
print(f"  转化率差: {obs_diff*100:.2f}个百分点")
print(f"  合并比例: {pooled_p*100:.2f}%")
print(f"  标准误 SE: {se*100:.4f}")
print(f"  95% CI: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")
print(f"  CI是否包含0: {'是(无显著差异)' if ci_lower <= 0 <= ci_upper else '否(有显著差异)'}")
print(f"  区间宽度: {(ci_upper - ci_lower)*100:.2f}个百分点")
if ci_lower <= 0 <= ci_upper:
    print(f"\n  解读: CI包含0，说明转化率差异在95%置信水平下不显著。")
    print(f"  可能原因: 样本量不足或真实差异较小。建议增加样本量继续实验。")
else:
    print(f"\n  解读: CI不包含0，说明转化率差异在95%置信水平下显著。")
    print(f"  CI下界{ci_lower*100:.2f}%是'最保守估计'的转化率提升。")
    print(f"  若CI下界 > 业务最小有效阈值(如0.5%)，则兼具统计和商业显著性。")

## 5. 卡方检验理论

### 卡方独立性检验
检验两个分类变量是否独立。构建列联表（contingency table），计算观察频数与期望频数的差异。

**统计量**：χ² = Σ (观察值 - 期望值)² / 期望值

**假设**：
- H₀: 两个变量独立（无关联）
- H₁: 两个变量有关联

### 营销应用
- 用户分群（新客/回客/VIP）与购买品类（美妆/电子/健身/家居）是否有关联？
- 如果有关联，可以做差异化推荐（VIP偏好电子→推送电子产品）
- 如果独立，无需按分群差异化策略

## 5：卡方检验 -- 用户分群与购买品类独立性

**任务**：用 scipy.stats.chi2_contingency 检验用户分群与购买品类是否独立。

**营销场景**：产品经理想知道"不同用户群体的品类偏好是否不同？"如果不同，可以按分群做差异化推荐策略。

**步骤**：
1. 筛选已转化用户（有购买品类的用户）
2. 用 pd.crosstab 构建列联表（segment × category）
3. 用 stats.chi2_contingency 执行检验
4. 用 seaborn.heatmap 可视化列联表

In [ ]:
# 5. 用 scipy.stats.chi2_contingency 执行卡方独立性检验
df_purchased = df[df['converted'] == 1].copy()
contingency = pd.crosstab(df_purchased['segment'], df_purchased['category'])

chi2, p_val_chi, dof, expected = stats.chi2_contingency(contingency)

print("--- 卡方独立性检验 ---")
print(f"  H0: 用户分群与购买品类独立")
print(f"  H1: 用户分群与购买品类有关联")
print(f"  列联表（实际频数）:")
print(contingency)
print(f"\n  卡方统计量: {chi2:.4f}")
print(f"  自由度: {dof}")
print(f"  p值: {p_val_chi:.4f}")
if p_val_chi < 0.05:
    print(f"  结论: 拒绝原假设。用户分群与购买品类有关联。")
    print(f"  商业含义: 不同用户群体的品类偏好不同，可做差异化推荐。")
else:
    print(f"  结论: 无法拒绝原假设。用户分群与购买品类独立。")
    print(f"  商业含义: 用户分群与品类偏好无关，无需差异化策略。")

# 热力图可视化
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(contingency, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('用户分群 × 购买品类 列联表')
ax.set_xlabel('购买品类')
ax.set_ylabel('用户分群')
plt.tight_layout()
plt.show()

## 6. 贝叶斯统计理论

### 频率派 vs 贝叶斯派

| 维度 | 频率派 | 贝叶斯派 |
|------|--------|---------|
| 概率定义 | 长期频率 | 不确定性程度 |
| 参数 | 固定但未知 | 随机变量，有分布 |
| 核心工具 | p值、置信区间 | 后验分布、可信区间 |
| 先验信息 | 不使用 | 显式融入 |
| 营销应用 | A/B测试标准流程 | 小样本估计、持续更新 |

### Beta-Binomial 模型
转化率 p 的贝叶斯推断最简模型：
- **先验**：Beta(α, β) — 表示在看到数据前对转化率的信念
- **似然**：Binomial(n, p) — 观察到 s 次成功 / n 次试验
- **后验**：Beta(α+s, β+n-s) — 更新后的信念

**后验均值** = (α+s) / (α+β+n)，随数据增加趋近真实转化率。

**无信息先验** Beta(1,1) = 均匀分布，不引入任何偏见。

### 贝叶斯优势
- 后验直接回答"转化率在X%-Y%之间的概率是95%"
- 小样本时通过先验给出更稳定估计
- 后验可随新数据持续更新（每次实验结果都是下次的先验）

### 概率编程 PyMC
理解 Beta-Binomial 原理后，可用 PyMC (https://www.pymc.io/) 扩展到层次贝叶斯模型，处理多层级结构（如不同渠道/用户群体的转化率有先验关联）。

## 6：贝叶斯统计 -- Beta-Binomial 后验估计

**任务**：用 Beta-Binomial 模型估计 B 组转化率的后验分布。

**营销场景**：你刚上线新版落地页，只有500次曝光。频率派 p 值可能不稳定，贝叶斯方法通过先验信息给出更合理的估计，并随着新数据持续更新。

**步骤**：
1. 设定先验 Beta(1,1)（无信息先验/均匀分布）
2. 从 B 组数据提取成功数和试验数
3. 计算后验 Beta(1+s, 1+n-s)
4. 计算后验均值和 95% 可信区间
5. 可视化先验→后验的更新过程
6. 与频率派结果对比

**提示**：用 scipy.stats.beta 的 .ppf() 计算分位点（credible interval）

In [ ]:
# 6. 用 Beta-Binomial 模型做贝叶斯转化率估计
from scipy.stats import beta as beta_dist

successes_b = int(conv_b.sum())
trials_b = len(conv_b)

prior_alpha = 1  # 无信息先验 Beta(1,1)
prior_beta = 1

# 后验参数更新：Beta(1+s, 1+n-s)
post_alpha = prior_alpha + successes_b
post_beta = prior_beta + trials_b - successes_b

# 后验统计量
post_mean = post_alpha / (post_alpha + post_beta)
ci_lower_bayes = beta_dist.ppf(0.025, post_alpha, post_beta)
ci_upper_bayes = beta_dist.ppf(0.975, post_alpha, post_beta)

# 频率派 B 组转化率的 95% CI（用于对比）
se_b = np.sqrt(conv_rate_b * (1 - conv_rate_b) / trials_b)
freq_ci_lower = conv_rate_b - 1.96 * se_b
freq_ci_upper = conv_rate_b + 1.96 * se_b

print("--- 贝叶斯统计：Beta-Binomial 后验估计（B组）---")
print(f"  先验: Beta({prior_alpha}, {prior_beta})（无信息先验/均匀分布）")
print(f"  数据: {successes_b}次转化 / {trials_b}次试验")
print(f"  后验: Beta({post_alpha}, {post_beta})")
print(f"  后验均值(转化率估计): {post_mean*100:.2f}%")
print(f"  95%可信区间: [{ci_lower_bayes*100:.2f}%, {ci_upper_bayes*100:.2f}%]")
print(f"\n  频率派 vs 贝叶斯派对比：")
print(f"    频率派点估计: {conv_rate_b*100:.2f}%")
print(f"    贝叶斯后验均值: {post_mean*100:.2f}%")
print(f"    频率派95%CI: [{freq_ci_lower*100:.2f}%, {freq_ci_upper*100:.2f}%]")
print(f"    贝叶斯95%可信区间: [{ci_lower_bayes*100:.2f}%, {ci_upper_bayes*100:.2f}%]")
print(f"    两者接近是因为样本量大、先验为无信息先验。")

# 可视化先验和后验
fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(0, 0.15, 1000)
ax.plot(x, beta_dist.pdf(x, prior_alpha, prior_beta), 'gray', linewidth=2, label=f'先验 Beta({prior_alpha},{prior_beta})')
ax.plot(x, beta_dist.pdf(x, post_alpha, post_beta), '#e74c3c', linewidth=2, label=f'后验 Beta({post_alpha},{post_beta})')
ax.axvline(post_mean, color='#e74c3c', linestyle='--', alpha=0.5, label=f'后验均值={post_mean*100:.2f}%')
ax.fill_between(x, beta_dist.pdf(x, post_alpha, post_beta),
                where=(x >= ci_lower_bayes) & (x <= ci_upper_bayes),
                alpha=0.2, color='#e74c3c', label='95%可信区间')
ax.set_xlabel('转化率')
ax.set_ylabel('概率密度')
ax.set_title('Beta-Binomial 模型：先验 -> 后验更新（B组转化率）')
ax.legend()
plt.tight_layout()
plt.savefig('bayesian_posterior.png', dpi=150)
plt.show()
print(f"\n  贝叶斯优势: 后验直接回答'转化率在X%-Y%之间的概率是95%'，")
print(f"  而频率派CI只能回答'重复实验100次，95次CI会包含真值'。")
print(f"\n  进阶: 用 PyMC (https://www.pymc.io/) 可扩展到层次贝叶斯模型，")
print(f"  对不同用户群体的转化率建模，处理小样本和多层级结构。")

## 7. 反思与前沿

### 反思问题
1. A/B 两组的客单价分布是右偏还是左偏？均值和中位数哪个更大？为什么？
2. t 检验的 p 值是多少？在 α=0.05 下是否显著？效应量有多大？
3. 95% 置信区间是否包含 0？下界是多少？这个下界的商业含义是什么？
4. 卡方检验结果是否表明用户分群与品类有关联？如果是，如何用于差异化推荐？
5. 贝叶斯后验均值与频率派点估计有多大差异？为什么？

### 2026 前沿：贝叶斯统计 + 概率编程 + 可复现研究

**贝叶斯统计复兴**：2020年代贝叶斯方法在营销场景中越来越重要：
- **小样本场景**：新产品上线初期曝光少，频率派p值不稳定，贝叶斯通过先验给出更合理估计
- **持续更新**：后验随新数据持续更新，天然适配"持续优化"的营销节奏
- **直接回答业务问题**：后验直接回答"方案有效的概率"，比频率派p值更贴近决策者直觉

**概率编程 PyMC**（https://www.pymc.io/）：Python最主流的贝叶斯推断框架，支持MCMC和变分推断。本Day用scipy.stats.beta手动实现Beta-Binomial理解原理后，可用PyMC扩展到层次贝叶斯模型。

**可复现研究**：统计学正经历"可复现危机"。p值操纵（p-hacking）是A/B测试中最常见的错误--不断检查p值，一旦显著就停止。应对措施：
- 预注册（preregistration）：实验前公开声明假设和样本量（OSF平台）
- 预计算样本量：用功效分析计算所需样本量，达到后再判断
- 报告效应量和CI：不仅报告p值，还报告效应量和置信区间

参考 [PyMC](https://www.pymc.io/) + [ASA p值声明](https://www.amstat.org/asa-statements) + [OSF预注册](https://osf.io/)。